# Result · Threshold sweep: is the ranking of defences an artifact of τ?

**In the paper ([arXiv:2610.10899](https://arxiv.org/abs/2610.10899)):** the τ sweep at the end of the Sec. V paragraph "Patching hardens narrowly".

Corrected-reference ASR of the fresh EnCodec attacker for τ from 0.1 to 1.0, with the median and 90th-percentile
score gap, per predictor (mean over seeds). Recomputes E1 (24 jobs, ~15 min each).

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters (also settable through environment variables for non-interactive runs)
# SOURCE = "recompute": re-run the evaluations behind this output from the released checkpoints (GPU), then aggregate.
# SOURCE = "released" : aggregate the released outputs of the original runs (CPU, seconds). This rewrites the
#                       committed analysis/out/ and paper/ files, which should come out identical.
SOURCE = os.environ.get("NB_SOURCE", "recompute")
# RUN = "slurm": one GPU job per run (sbatch), this kernel waits; "local": run in this kernel (needs a GPU).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e1']]
print(len(KEYS), "runs:", ", ".join(KEYS))

24 runs: e1_R0_s0, e1_R3_s0, e1_R0_s1, e1_R3_s1, e1_R0_s2, e1_R3_s2, e1_baseline_noise_s0, e1_baseline_pgd_s0, e1_baseline_random_s0, e1_baseline_noise_s1, e1_baseline_pgd_s1, e1_baseline_random_s1, e1_baseline_noise_s2, e1_baseline_pgd_s2, e1_baseline_random_s2, e1_R1_s0, e1_R2_s0, e1_R1_s1, e1_R2_s1, e1_R1_s2, e1_R2_s2, e1_R3ms_s0, e1_R3ms_s1, e1_R3ms_s2


In [4]:
if SOURCE == "recompute":
    repro.run_keys(KEYS, "recompute", RUN)  # skips runs whose recomputed output already exists
dirs = repro.run_analysis(SOURCE)  # whole analysis chain (seconds); falls back to released outputs where needed
OUT, PAPER = dirs["out"], dirs["paper"]
COMMITTED_OUT, COMMITTED_PAPER = ROOT / "analysis" / "out", ROOT / "paper"

[analysis] runs=/fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/runs
[analysis] out=/users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
[analysis] paper=/users/PAS2301/alialavi/projects/audio_assessment_hackability/paper


In [5]:
# Where this notebook's inputs came from: recomputed from checkpoints, or released outputs of the original runs.
used = repro.load_registry(repro.results_registry(SOURCE))
pd.Series({k: used[k].get("source", "released") for k in KEYS}, name="source").value_counts()

source
released    24
Name: count, dtype: int64

In [6]:
display(Markdown((OUT / "e1_tau_sweep.md").read_text()))

# Corrected-reference ASR vs detection threshold (fresh EnCodec attacker, mean over seeds)

| target          |   0.1 |   0.2 |   0.3 |   0.5 |   0.75 |   1.0 |   median_gap |   p90_gap |
|:----------------|------:|------:|------:|------:|-------:|------:|-------------:|----------:|
| R0              | 0.481 | 0.358 | 0.226 | 0.069 |  0.005 | 0     |        0.12  |     0.359 |
| R1              | 0.185 | 0.118 | 0.05  | 0.006 |  0     | 0     |        0.135 |     0.323 |
| R2              | 0.112 | 0.051 | 0.017 | 0.002 |  0     | 0     |        0.021 |     0.203 |
| R3              | 0.244 | 0.183 | 0.137 | 0.098 |  0.068 | 0.045 |        0.125 |     0.651 |
| R3ms            | 0.182 | 0.126 | 0.082 | 0.037 |  0.01  | 0.003 |       -0.495 |     0.135 |
| baseline_noise  | 0.63  | 0.447 | 0.27  | 0.071 |  0.009 | 0     |        0.203 |     0.465 |
| baseline_pgd    | 0.573 | 0.404 | 0.223 | 0.047 |  0.004 | 0     |        0.198 |     0.432 |
| baseline_random | 0.117 | 0.061 | 0.019 | 0.001 |  0     | 0     |        0.104 |     0.271 |


In [7]:
repro.compare_csv(OUT / "e1_tau_sweep.csv", COMMITTED_OUT / "e1_tau_sweep.csv")

seed    0.0
asr     0.0
Name: max_abs_diff, dtype: float64